# Fetch data

In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
y = y.iloc[:, 0]

# Prepare attributes, pipeline and split sets

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

# NaNN and NaN --> np.nan
X = X.replace(["NaN", "NaNN", "", " "], np.nan)

# Change categorical variables to numeric
cols_to_numeric = ["Cholesterol", "Copper", "Tryglicerides", "Platelets"]
X[cols_to_numeric] = X[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# Change Stage to category
X["Stage"] = X["Stage"].astype("category")

# Split data into training and test sets
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

# Cross validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

# Preprocessor
cat_cols = X_rest.select_dtypes(include=["object", "str", "category"]).columns
num_cols = X_rest.select_dtypes(include=["number"]).columns

# Preprocessor with imputation
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ]
)

# Bayes classificator

In [6]:

from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

model = Pipeline([
    ("prep", preprocessor),
    ("nb", GaussianNB())
])

param_grid = {
    # GaussianNB
    'nb__var_smoothing': np.logspace(0, -12, num=100),
    # Preprocessing
    'prep__num__strategy': ['mean', 'median'],
    'prep__cat__imputer__strategy': ['most_frequent'] 
}

grid_search = GridSearchCV(
    estimator=model, 
    param_grid=param_grid, 
    cv=cv, 
    scoring="accuracy", 
    n_jobs=-1, # All available cores
)

grid_search.fit(X_rest, y_rest)

print("Best params")
for param_name in sorted(param_grid.keys()):
    print(f"\t{param_name}: {grid_search.best_params_[param_name]}")
print(f"\nBest Accuracy from cv: {grid_search.best_score_:.4f}\n")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Test Set (Zoptymalizowany Model) - Classification Report\n")
print(classification_report(y_test, y_pred_best))


Best params
	nb__var_smoothing: 7.56463327554629e-09
	prep__cat__imputer__strategy: most_frequent
	prep__num__strategy: mean

Best Accuracy from cv: 0.7182

Test Set (Zoptymalizowany Model) - Classification Report

              precision    recall  f1-score   support

           C       0.71      0.87      0.78        47
          CL       0.00      0.00      0.00         5
           D       0.70      0.44      0.54        32

    accuracy                           0.65        84
   macro avg       0.47      0.44      0.44        84
weighted avg       0.66      0.65      0.64        84

